<a href="https://colab.research.google.com/github/0xfffddd/Coding/blob/main/part_D_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Part D — Qwen (Model Name: qwen3-vl-flash-2025-10-15-u) Multimodal LLM Agent CoT Pipeline
# Train & validation dataset: history dataset
# TEST: new dataset

#step0. import libary
!pip -q install pandas requests tqdm pillow numpy scikit-learn
import os, re, json, time, hashlib, base64
from io import BytesIO
from typing import Dict, Any, List, Tuple, Optional
import numpy as np
import pandas as pd
import requests
from tqdm import tqdm
import getpass
from PIL import Image
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

#step1. define and list files path for train and test
HIST_CSV  = "/content/dataset_history.csv"
HIST_ZIP  = "/content/image.zip"
NEW_CSV   = "/content/dataset_new.csv"
NEW_ZIP   = "/content/image_new.zip"
BASE_DIR  = "/content"
OUT_HIST  = "/content/partD_history_predictions.csv"
OUT_NEW   = "/content/partD_new_predictions.csv"
CACHE_DIR = "/content/llm_cache_partD_qwen"
os.makedirs(CACHE_DIR, exist_ok=True)

#step2. define the Multimodal LLM Agent and related parameters
BASE_URL = "https://dashscope-us.aliyuncs.com/compatible-mode/v1"
CHAT_URL = f"{BASE_URL}/chat/completions"
MODEL = "qwen3-vl-flash-2025-10-15-us"
  # token usage control
MAX_TOKENS   = 260
MAX_SIDE     = 384
JPEG_QUALITY = 80
  # retry controls
MAX_RETRIES = 6
BASE_SLEEP  = 1.0
  # self-consistency
N_SAMPLES   = 3
TEMPERATURE = 0.2
  # evaluation + selection
TOP_K = 20
USE_THRESHOLD = True

#step3. dataset parameter engineering
IMAGE_PATH_COL = "image_path"
TITLE_COL      = "title"
PRICE_COL      = "price"
LABEL_COL      = "ordered"
DIM_KEYS = ["visual_clarity","perceived_quality","style_trend","price_fairness","trust_signals"]

#step4. unzip the image files (if the images uploaded with zip files)
def unzip_if_needed(zip_path: str, out_dir: str = BASE_DIR):
    if os.path.exists(zip_path):
        print(f"Unzipping {zip_path} to {out_dir} ...")
        !unzip -q -o "$zip_path" -d "$out_dir"
    else:
        print(f"WARNING: zip not found: {zip_path} (skip unzip)")

#step5 type in API key to start the LLM
API_KEY = getpass.getpass("Enter your Qwen API Key (input hidden): ").strip()
assert API_KEY, "API Key is empty."
HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

#step6. write prompts for LLM
PROMPT_COT = """
You are an AI shopping agent simulating a human fashion buyer.

Goal: Predict purchase likelihood and decide whether to SELECT this product for promotion.

Input:
Title: {title}
Price: {price}

Reasoning strategy (do NOT output long chain-of-thought):
Briefly evaluate using these dimensions:
1) visual_clarity
2) perceived_quality
3) style_trend
4) price_fairness
5) trust_signals

Output rules:
- Return ONLY valid JSON.
- p_purchase: number in [0,1] with 2 decimals.
- decision: "select" or "reject".
- dimension_scores: integers 0-100 for the 5 dimensions.
- rationales: 2-4 short bullets referencing visible cues/title/price.
- risks: 0-2 short bullets.
- confidence: number in [0,1] with 2 decimals.

JSON schema:
{{
  "p_purchase": 0.00,
  "decision": "select",
  "dimension_scores": {{
    "visual_clarity": 0,
    "perceived_quality": 0,
    "style_trend": 0,
    "price_fairness": 0,
    "trust_signals": 0
  }},
  "rationales": ["..."],
  "risks": ["..."],
  "confidence": 0.00
}}
""".strip()

#step7 pre-processing on image
def normalize_to_abs(path_str: str, base_dir: str = BASE_DIR) -> str:
    p = str(path_str).strip().replace("\\", "/")
    if os.path.isabs(p):
        return p
    return os.path.join(base_dir, p)

def image_to_data_url_resized(path: str, max_side: int = MAX_SIDE, quality: int = JPEG_QUALITY) -> str:
    img = Image.open(path).convert("RGB")
    w, h = img.size
    scale = min(max_side / max(w, h), 1.0)
    if scale < 1.0:
        img = img.resize((int(w * scale), int(h * scale)))
    buf = BytesIO()
    img.save(buf, format="JPEG", quality=quality, optimize=True)
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{b64}"

#step8. use API to train - input (retry process will exetute when failed to use API)
def build_payload(prompt: str, data_url: str, temperature: float) -> Dict[str, Any]:
    return {
        "model": MODEL,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": data_url}},
            ],
        }],
        "max_tokens": MAX_TOKENS,
        "temperature": temperature,
    }

def call_with_retry(payload: Dict[str, Any]) -> Dict[str, Any]:
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.post(CHAT_URL, headers=HEADERS, json=payload, timeout=120)
            try:
                data = r.json()
            except Exception:
                data = {"raw_text": r.text}
            err = (data or {}).get("error", {})
            if err.get("code") == "Arrearage" or err.get("type") == "Arrearage":
                raise RuntimeError(f"ACCOUNT_ARREARAGE: {err.get('message')}")
            if r.status_code == 200:
                return data
            if r.status_code in (429, 500, 502, 503, 504, 408, 409):
                sleep_s = min(BASE_SLEEP * (2 ** (attempt - 1)), 30) + 0.15 * attempt
                time.sleep(sleep_s)
                continue
            raise RuntimeError(f"HTTP {r.status_code}: {data}")

        except RuntimeError as e:
            if "ACCOUNT_ARREARAGE" in str(e):
                raise
            last_err = e
            sleep_s = min(BASE_SLEEP * (2 ** (attempt - 1)), 30) + 0.15 * attempt
            time.sleep(sleep_s)
        except Exception as e:
            last_err = e
            sleep_s = min(BASE_SLEEP * (2 ** (attempt - 1)), 30) + 0.15 * attempt
            time.sleep(sleep_s)

    raise RuntimeError(f"Failed after {MAX_RETRIES} retries. Last error: {last_err}")

def extract_text(resp: Dict[str, Any]) -> str:
    msg = resp.get("choices", [{}])[0].get("message", {})
    c = msg.get("content", "")
    if isinstance(c, str):
        return c
    if isinstance(c, list):
        parts = []
        for blk in c:
            if isinstance(blk, dict):
                if "text" in blk:
                    parts.append(str(blk["text"]))
                elif blk.get("type") == "text" and "content" in blk:
                    parts.append(str(blk["content"]))
        return "\n".join(parts).strip()
    return str(c)

#step8. use API to train - output (with JSON files and the process for parse)
def _strip_to_json(text: str) -> str:
    t = (text or "").strip()
    t = re.sub(r"```json|```", "", t).strip()
    m = re.search(r"\{.*\}", t, flags=re.DOTALL)
    if m:
        t = m.group(0).strip()
    return t
def parse_cot_json(text: str) -> Dict[str, Any]:
    if not text:
        raise ValueError("Empty response text")
    t = _strip_to_json(text)
    obj = json.loads(t)
    # p_purchase
    p = obj.get("p_purchase", None)
    try:
        p = float(p)
        p = max(0.0, min(1.0, p))
        p = round(p, 2)
    except Exception:
        p = None
    # decision
    decision = str(obj.get("decision","")).strip().lower()
    if decision not in ("select","reject"):
        decision = None
    # confidence
    conf = obj.get("confidence", None)
    try:
        conf = float(conf)
        conf = max(0.0, min(1.0, conf))
        conf = round(conf, 2)
    except Exception:
        conf = None
    # rationales / risks
    rats = obj.get("rationales", [])
    if isinstance(rats, str):
        rats = [rats]
    if not isinstance(rats, list):
        rats = []
    rats = [str(x).strip() for x in rats if str(x).strip()][:4]
    risks = obj.get("risks", [])
    if isinstance(risks, str):
        risks = [risks]
    if not isinstance(risks, list):
        risks = []
    risks = [str(x).strip() for x in risks if str(x).strip()][:2]
    # dimension_scores
    dims = obj.get("dimension_scores", {}) or {}
    if not isinstance(dims, dict):
        dims = {}
    dd = {}
    for k in DIM_KEYS:
        v = dims.get(k, None)
        try:
            iv = int(v)
            iv = max(0, min(100, iv))
        except Exception:
            iv = None
        dd[k] = iv
    return {
        "p_purchase": p,
        "decision": decision,
        "dimension_scores": dd,
        "rationales": rats,
        "risks": risks,
        "confidence": conf,
    }

#step9. write local cache to handle interrupt or accident (the train process can continue on the previous progress without re-training)
def make_cache_key(dataset_tag: str, title: str, price: Any, abs_image_path: str) -> str:
    raw = f"{dataset_tag}|{MODEL}|{title}|{price}|{abs_image_path}|cot"
    return hashlib.md5(raw.encode("utf-8")).hexdigest()
def cache_fp(key: str) -> str:
    return os.path.join(CACHE_DIR, f"{key}.json")
def cache_get(key: str):
    fp = cache_fp(key)
    if os.path.exists(fp):
        with open(fp, "r", encoding="utf-8") as f:
            return json.load(f)
    return None
def cache_set(key: str, obj: dict):
    fp = cache_fp(key)
    with open(fp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)

#step10. run_one_item: runs the multimodal LLM on a single product N times (self-consistency),
#     then aggregates results into one final p_purchase/decision plus merged rationales
#     and caches the output (to reduce randomness and avoid repeat API cost).
def run_one_item(abs_path: str, title: str, price: Any, dataset_tag: str) -> Tuple[Dict[str, Any], str]:
    if not os.path.exists(abs_path):
        return ({
            "p_purchase": None,
            "decision": None,
            "dimension_scores": {k: None for k in DIM_KEYS},
            "rationales": [],
            "risks": [],
            "confidence": None,
            "_cached": 0
        }, "missing_file")
    ck = make_cache_key(dataset_tag, title, price, abs_path)
    cached = cache_get(ck)
    if cached is not None:
        cached["_cached"] = 1
        return cached, ""
    try:
        data_url = image_to_data_url_resized(abs_path)
        prompt = PROMPT_COT.format(title=title, price=price)

        parsed_list = []
        for _ in range(N_SAMPLES):
            payload = build_payload(prompt, data_url, TEMPERATURE)
            resp = call_with_retry(payload)
            txt = extract_text(resp)
            parsed_list.append(parse_cot_json(txt))

        ps = [x["p_purchase"] for x in parsed_list if x.get("p_purchase") is not None]
        p_mean = round(float(np.mean(ps)), 2) if ps else None
        # decision majority
        decisions = [x["decision"] for x in parsed_list if x.get("decision") in ("select","reject")]
        if decisions:
            sel = sum(d == "select" for d in decisions)
            rej = sum(d == "reject" for d in decisions)
            if sel > rej:
                d_final = "select"
            elif rej > sel:
                d_final = "reject"
            else:
                d_final = "select" if (p_mean is not None and p_mean >= 0.5) else "reject"
        else:
            d_final = None
        # dims mean
        dims_agg = {}
        for k in DIM_KEYS:
            vals = [x["dimension_scores"].get(k) for x in parsed_list if x.get("dimension_scores", {}).get(k) is not None]
            dims_agg[k] = int(round(np.mean(vals))) if vals else None
        # union rationales/risks
        rats = []
        for x in parsed_list:
            for r in x.get("rationales", []):
                if r and r not in rats:
                    rats.append(r)
        rats = rats[:4]
        risks = []
        for x in parsed_list:
            for r in x.get("risks", []):
                if r and r not in risks:
                    risks.append(r)
        risks = risks[:2]
        confs = [x["confidence"] for x in parsed_list if x.get("confidence") is not None]
        conf_mean = round(float(np.mean(confs)), 2) if confs else None
        out = {
            "p_purchase": p_mean,
            "decision": d_final,
            "dimension_scores": dims_agg,
            "rationales": rats,
            "risks": risks,
            "confidence": conf_mean,
            "_cached": 0
        }
        cache_set(ck, out)
        return out, ""
    except Exception as e:
        return ({
            "p_purchase": None,
            "decision": None,
            "dimension_scores": {k: None for k in DIM_KEYS},
            "rationales": [],
            "risks": [],
            "confidence": None,
            "_cached": 0
        }, str(e)[:2000])

#step11. run_dataset - Processes an entire history data CSV end-to-end by loading data, resolving image paths,
#     skipping already-completed rows, calling run_one_item per product,
#     and writing predictions to disk in chunks (resume-safe batch inference).
def run_dataset(csv_path: str, zip_path: str, out_path: str, dataset_tag: str) -> pd.DataFrame:
    unzip_if_needed(zip_path, BASE_DIR)
    assert os.path.exists(csv_path), f"CSV not found: {csv_path}"
    d = pd.read_csv(csv_path)
    d["abs_image_path"] = d[IMAGE_PATH_COL].astype(str).apply(normalize_to_abs)
    # resume-safe: only treat rows as done if p_purchase is present
    done_set = set()
    if os.path.exists(out_path):
        prev = pd.read_csv(out_path)
        if "abs_image_path" in prev.columns and "p_purchase" in prev.columns:
            done_set = set(prev.loc[prev["p_purchase"].notna(), "abs_image_path"].astype(str).tolist())
        print(f"Resume: found {len(done_set)} valid completed rows in {out_path}")

    buffer = []
    missing = 0
    errors  = 0
    def append_rows(rows: list):
        if not rows:
            return
        df_chunk = pd.DataFrame(rows)
        write_header = not os.path.exists(out_path)
        df_chunk.to_csv(out_path, mode="a", header=write_header, index=False)
    for _, row in tqdm(d.iterrows(), total=len(d), desc=f"Part D inference ({dataset_tag})"):
        abs_path = str(row["abs_image_path"])
        if abs_path in done_set:
            continue
        title = str(row.get(TITLE_COL, ""))
        price = row.get(PRICE_COL, "")
        out_row = dict(row)
        out_row["_model"] = MODEL
        out_row["_error"] = None
        res, err = run_one_item(abs_path, title, price, dataset_tag=dataset_tag)
        out_row["p_purchase"] = res.get("p_purchase")
        out_row["decision"]   = res.get("decision")
        out_row["confidence"] = res.get("confidence")
        out_row["rationales"] = json.dumps(res.get("rationales", []), ensure_ascii=False)
        out_row["risks"]      = json.dumps(res.get("risks", []), ensure_ascii=False)
        out_row["_cached"]    = res.get("_cached", 0)
        dims = res.get("dimension_scores", {}) or {}
        for k in DIM_KEYS:
            out_row[f"dim_{k}"] = dims.get(k)
        if err:
            out_row["_error"] = err
            if err == "missing_file":
                missing += 1
            else:
                errors += 1
        buffer.append(out_row)
        if len(buffer) >= 50:
            append_rows(buffer)
            buffer = []
    append_rows(buffer)
    print(f"\nSaved: {out_path}")
    print("Missing files:", missing, "| Errors:", errors)
    return pd.read_csv(out_path)

# step12. define metrics for reasoning and validations
def precision_at_k(y_true: np.ndarray, y_score: np.ndarray, k: int) -> float:
    k = max(1, min(int(k), len(y_true)))
    idx = np.argsort(-y_score)[:k]
    return float(np.mean(y_true[idx]))
def evaluate(df_pred: pd.DataFrame, name: str, top_k: int, threshold: Optional[float]):
    dfv = df_pred.copy()
    if "p_purchase" not in dfv.columns or LABEL_COL not in dfv.columns:
        print(f"[{name}] Missing columns for evaluation. Skip.")
        return
    dfv = dfv[dfv["p_purchase"].notna()]
    dfv = dfv[dfv[LABEL_COL].notna()]
    if len(dfv) == 0:
        print(f"[{name}] N(valid)=0, skip.")
        return
    y_true = dfv[LABEL_COL].astype(int).values
    y_score = dfv["p_purchase"].astype(float).values
    auc = None
    if len(np.unique(y_true)) == 2:
        auc = roc_auc_score(y_true, y_score)
    p_at_k = precision_at_k(y_true, y_score, top_k)
    print(f"\n=== Evaluation: {name} ===")
    print("N (valid):", len(dfv))
    print(f"AUC: {auc if auc is not None else 'N/A (single class)'}")
    print(f"Precision@{min(top_k, len(dfv))}: {p_at_k:.4f}")
    if threshold is not None:
        y_hat = (y_score >= threshold).astype(int)
        prec = precision_score(y_true, y_hat, zero_division=0)
        rec  = recall_score(y_true, y_hat, zero_division=0)
        f1   = f1_score(y_true, y_hat, zero_division=0)
        sel_rate = float(np.mean(y_hat))
        print(f"Threshold: {threshold:.2f} | Selected rate: {sel_rate:.3f}")
        print(f"Precision(thr): {prec:.4f} | Recall(thr): {rec:.4f} | F1(thr): {f1:.4f}")
def calibrate_threshold_on_history(df_hist: pd.DataFrame) -> float:
    dfv = df_hist.copy()
    dfv = dfv[dfv["p_purchase"].notna()]
    dfv = dfv[dfv[LABEL_COL].notna()]
    if len(dfv) == 0:
        return 0.5
    y_true = dfv[LABEL_COL].astype(int).values
    y_score = dfv["p_purchase"].astype(float).values
    cands = np.round(np.arange(0.10, 0.91, 0.05), 2)
    best_thr, best_f1 = 0.50, -1.0
    for t in cands:
        y_hat = (y_score >= t).astype(int)
        f1 = f1_score(y_true, y_hat, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(t)
    print("\n=== Threshold calibration on HISTORY ===")
    print("Best threshold (by F1):", best_thr, "| Best F1:", round(best_f1, 4))
    return best_thr

# step13. main function assemble: train LLM model based on history dataset, and predict on the new dataset
#      then do the validataion for the model on the history dataset
print("\n[1/2] Running HISTORY (dev) inference ...")
hist_pred = run_dataset(HIST_CSV, HIST_ZIP, OUT_HIST, dataset_tag="history")
evaluate(hist_pred, "HISTORY (dev)", top_k=TOP_K, threshold=None)
thr = None
if USE_THRESHOLD:
    thr = calibrate_threshold_on_history(hist_pred)
    evaluate(hist_pred, "HISTORY (dev) with threshold", top_k=TOP_K, threshold=thr)
print("\n[2/2] Running NEW (test) inference ...")
new_pred = run_dataset(NEW_CSV, NEW_ZIP, OUT_NEW, dataset_tag="new")
  # skip null columns ( new.csv may not have labels then print no evaluation if ordered missing)
if LABEL_COL in new_pred.columns:
    evaluate(new_pred, "NEW (test)", top_k=TOP_K, threshold=thr if USE_THRESHOLD else None)
else:
    print(f"[NEW (test)] No label column '{LABEL_COL}' found. Inference completed (blind test).")

print("\nDone.")
print("History predictions:", OUT_HIST)
print("New predictions:", OUT_NEW)


[1/2] Running HISTORY (dev) inference ...
Unzipping /content/image.zip to /content ...


Part D inference (history): 100%|██████████| 331/331 [29:10<00:00,  5.29s/it]



Saved: /content/partD_history_predictions.csv
Missing files: 1 | Errors: 0

=== Evaluation: HISTORY (dev) ===
N (valid): 330
AUC: 0.5062222222222222
Precision@20: 0.4500

=== Threshold calibration on HISTORY ===
Best threshold (by F1): 0.55 | Best F1: 0.4895

=== Evaluation: HISTORY (dev) with threshold ===
N (valid): 330
AUC: 0.5062222222222222
Precision@20: 0.4500
Threshold: 0.55 | Selected rate: 0.982
Precision(thr): 0.3241 | Recall(thr): 1.0000 | F1(thr): 0.4895

[2/2] Running NEW (test) inference ...
Unzipping /content/image_new.zip to /content ...


Part D inference (new): 100%|██████████| 5/5 [00:26<00:00,  5.26s/it]


Saved: /content/partD_new_predictions.csv
Missing files: 0 | Errors: 0
[NEW (test)] No label column 'ordered' found. Inference completed (blind test).

Done.
History predictions: /content/partD_history_predictions.csv
New predictions: /content/partD_new_predictions.csv


In [ ]:
# Code to clean the cache if the upper cell bugged and have to run again (DO NOT RUN IF upper cell successed, it will train and test the whole model again)
# To run this cell, delete '#' for the following lines:

#!rm -f /content/partD_history_cot_predictions.csv
#!rm -f /content/partD_new_cot_predictions.csv
#!rm -rf /content/llm_cache_partD_qwen